# Batch 2 — E19, E20, E11b, E14 on the new parent (validation only)

**Runtime:** Runtime → Change runtime type → **L4 GPU**. Then Runtime → Run all; about **1 h 45 min – 2 h**. Keep the tab open.
If Colab disconnects, Run all again: finished runs are skipped.

**Parent = E10 + E12** (batch 1): E03 configuration with RevIN **off**, each horizon scored with its own best epoch.
Its batch 1 file (`results/batch1/e10_no_revin.json`) is the reference; it is not rerun, because batch 1 reproduced E03 exactly.
Every run below is the parent with **one** change (fixed before any run, `claude/batch2-design-2026-10-16.md`):

| Run | Change vs parent | Order |
|---|---|---|
| E19 | stop only when **every** horizon has stalled for 10 epochs (1-day head was cut off in batch 1) | 1st; its first epochs must match the parent run exactly (free reproducibility check) |
| E20 | walk-forward check of the parent: same 4 folds as E05 (XGBoost fold files reused from E05) + the official window | 2nd |
| E11b | classification only (no return-forecast loss) | 3rd |
| E14 | look-back 120 trading days instead of 60 | 4th (slowest) |

Seeds 20261015/16/17. Outputs in `MyDrive/Senior/results/batch2/`. Test block stays sealed.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib, os
src = open('scripts/baselines_deep.py', encoding='utf-8').read()
assert '--stop-rule' in src and '--lookback' in src and 'es_per_horizon' in src, 'stop: push the latest scripts/baselines_deep.py from your PC'
assert os.path.exists('src/stopping.py') and os.path.exists('tests/test_stopping.py'), 'stop: push src/stopping.py and tests/test_stopping.py'
assert '--per-horizon-best' in open('scripts/compare_batch.py', encoding='utf-8').read(), 'stop: push the latest scripts/compare_batch.py'
assert '--per-horizon-best' in open('scripts/summarize_walk_forward.py', encoding='utf-8').read(), 'stop: push the latest scripts/summarize_walk_forward.py'
for f in ['scripts/baselines_deep.py', 'src/stopping.py', 'scripts/compare_batch.py', 'scripts/summarize_walk_forward.py']:
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
# Files from earlier batches that this batch compares against
PARENT = f'{OUT_DIR}/batch1/e10_no_revin.json'
FOLDS = [('2022H2', '2022-07-01', '2022-12-31'), ('2023H1', '2023-01-01', '2023-06-30'),
         ('2023H2', '2023-07-01', '2023-12-31'), ('2024H1', '2024-01-01', '2024-06-30')]
need = [PARENT, f'{OUT_DIR}/xgb_e03.json'] + [f'{OUT_DIR}/e05/{f}_xgb.json' for f, _, _ in FOLDS]
for f in need:
    assert os.path.exists(f), f'stop: missing {f}'
B2 = f'{OUT_DIR}/batch2'
os.makedirs(f'{B2}/e20', exist_ok=True)
print('earlier files found; outputs go to', B2)

In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo "$OUT_DIR"
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
# Files must match the PC copies (hashes recorded 2026-10-15; sentiment file as stored in git, LF line endings)
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
# Pipeline, look-ahead, walk-forward and stop-rule tests; the notebook stops here if any fails (about 5 minutes)
import subprocess
for t in ['tests/test_pipeline.py', 'tests/test_no_lookahead.py', 'tests/test_walk_forward.py', 'tests/test_stopping.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l)[-1500:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed'

In [ ]:
# Switch check (stops on mismatch): weight counts with RevIN off at look-back 60 and 120, output shapes
import sys, torch
sys.path.insert(0, 'scripts'); sys.path.insert(0, 'src')
import baselines_deep as BD
def n(m): return sum(p.numel() for p in m.parameters())
BD.OPT.update(revin=False, cls_only=False, label_smoothing=0.0, stop_rule='mean')
m60 = BD.make_model('multiscale_sector', 32, 18, 0.2)
BD.LOOKBACK = 120
m120 = BD.make_model('multiscale_sector', 32, 18, 0.2)
out = m120(torch.randn(8, 120, 32), torch.randint(0, 19, (8,)))
BD.LOOKBACK = 60
print('weights, RevIN off: look-back 60', n(m60), '| look-back 120', n(m120))
# 120 days: 24 + 11 tokens instead of 12 + 5 -> joining layer +18*64*64, position tables +18*64
assert n(m60) == 472_572 and n(m120) == 472_572 + 18 * 64 * 64 + 18 * 64 == 547_452, 'stop: weight count differs from the design'
assert m120.revin is None and tuple(out['trend_logits'].shape) == (8, 3, 3) and tuple(out['returns'].shape) == (8, 3), 'stop: output shape'
print('switch check passed')

In [ ]:
# Smoke run (2 epochs, 1 seed, scratch file) with every new switch at once: look-back 120, stop rule 'every', RevIN off
import json, subprocess
subprocess.run(['python', 'scripts/baselines_deep.py', '--models', 'multiscale_sector', '--news', 'data/derived/sentiment_scores.csv',
                '--extra-features', 'e03', '--lr', '1e-4', '--dropout', '0.2', '--max-epochs', '2', '--patience', '1',
                '--no-revin', '--stop-rule', 'every', '--lookback', '120', '--seeds', '20261015', '--out', '/content/smoke.json'],
               check=True, capture_output=True)
res = json.load(open('/content/smoke.json'))
run = res['models']['multiscale_sector']['runs'][0]
assert res['settings']['lookback'] == 120 and res['settings']['options']['stop_rule'] == 'every' and not res['settings']['options']['revin']
assert run['params'] == 547_452 and 'es_per_horizon' in run['history'][0] and 'epochs_run' in run, 'stop: smoke output incomplete'
print('smoke run passed:', run['epochs_run'], 'epochs, best epoch per horizon', run['best_epoch_per_horizon'])

In [ ]:
import json, subprocess, sys
COMMON = ['--models', 'multiscale_sector', '--news', 'data/derived/sentiment_scores.csv', '--extra-features', 'e03',
          '--lr', '1e-4', '--dropout', '0.2', '--max-epochs', '30', '--patience', '10', '--no-revin']   # parent = E10 (+ E12 scoring)
def done(path):
    try:
        return len(json.load(open(path))['models']['multiscale_sector']['runs']) == 3
    except Exception:
        return False
def run(label, extra, out):
    if done(out):
        print(f'=== {label}: already done'); return
    print(f'=== {label}'); sys.stdout.flush()
    subprocess.run(['python', 'scripts/baselines_deep.py', *COMMON, *extra, '--out', out], check=True)

# 1. E19: stop only when every horizon has stalled
run('E19 every-horizon stop', ['--stop-rule', 'every'], f'{B2}/e19_every_horizon.json')

In [ ]:
# Reproducibility check (free): E19 trains exactly like the parent until the parent stopped, so those epochs must match.
p = {r['seed']: r for r in json.load(open(PARENT))['models']['multiscale_sector']['runs']}
diffs = []
for r in json.load(open(f'{B2}/e19_every_horizon.json'))['models']['multiscale_sector']['runs']:
    a, b = p[r['seed']]['history'], r['history']
    k = len(a)
    d = max(abs(x['es_macro_f1'] - y['es_macro_f1']) for x, y in zip(a, b[:k]))
    diffs.append(d)
    print(f"seed {r['seed']}: parent stopped at epoch {k}, E19 at epoch {r['epochs_run']}; "
          f"max |difference| over epochs 1-{k}: {d:.4f}; best epoch per horizon {r['best_epoch_per_horizon']}")
print('REPRODUCED EXACTLY' if max(diffs) == 0 else 'NOTE: not bit-identical to batch 1 - tell Claude (results still usable, judged with care)')

In [ ]:
# 2. E20: walk-forward check of the parent. Deep model per fold; XGBoost fold files are reused from E05
#    (XGBoost does not depend on RevIN; summarize_walk_forward.py asserts the folds and rows match).
for label, es, ee in FOLDS:
    run(f'E20 {label}', ['--eval-start', es, '--eval-end', ee], f'{B2}/e20/{label}_deep.json')

In [ ]:
# 3. E11b: classification only, on the parent
run('E11b classification only', ['--cls-only'], f'{B2}/e11b_cls_only.json')
# 4. E14: look-back 120 trading days (about 2x slower per epoch)
run('E14 look-back 120', ['--lookback', '120'], f'{B2}/e14_lookback120.json')

In [ ]:
# Summary 1: E19, E11b, E14 against the parent, all scored with each horizon's own best epoch (E12)
!python scripts/compare_batch.py --per-horizon-best --control-label "E10+E12" --control "$OUT_DIR/batch1/e10_no_revin.json" \
    --run E19 "$OUT_DIR/batch2/e19_every_horizon.json" --run E11b "$OUT_DIR/batch2/e11b_cls_only.json" \
    --run E14 "$OUT_DIR/batch2/e14_lookback120.json" --out "$OUT_DIR/batch2/batch2_summary.json"

In [ ]:
# Summary 2: E20 walk-forward of the parent. First with per-horizon best epochs (the parent's scoring rule),
# then with the shared best epoch (directly comparable with E05, which predates per-horizon scoring).
WF = ' '.join(f'--fold {f} "{OUT_DIR}/batch2/e20/{f}_deep.json" "{OUT_DIR}/e05/{f}_xgb.json"' for f, _, _ in FOLDS)
WF += f' --fold official "{OUT_DIR}/batch1/e10_no_revin.json" "{OUT_DIR}/xgb_e03.json"'
!python scripts/summarize_walk_forward.py --experiment E20 --per-horizon-best {WF} --out "$OUT_DIR/batch2/e20_summary_per_horizon.json"
!python scripts/summarize_walk_forward.py --experiment E20 {WF} --out "$OUT_DIR/batch2/e20_summary_shared_epoch.json"